# Model Experiment — XGBoost

End-to-end notebook for **XGBoost** on the IEEE-CIS fraud dataset.
Each section is logged as a separate MLflow run inside the
`XGBoost_Training` experiment.


## 0. Setup

First cell: bootstraps Kaggle secrets / data paths.
Second cell: inlined `preprocessing.py` (so this notebook is self-contained on Kaggle).
Third cell: imports + DagsHub MLflow init.

In [ ]:
# --- Kaggle / local bootstrap ---------------------------------------------
# pulls MLflow creds from Kaggle Secrets when running on Kaggle, else falls
# back to env vars. data path auto-detects /kaggle/input/ieee-fraud-detection.
import os, sys, gc, warnings
warnings.filterwarnings("ignore")

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        sec = UserSecretsClient()
        os.environ["MLFLOW_TRACKING_USERNAME"] = sec.get_secret("MLFLOW_TRACKING_USERNAME")
        os.environ["MLFLOW_TRACKING_PASSWORD"] = sec.get_secret("MLFLOW_TRACKING_PASSWORD")
    except Exception as e:
        print("kaggle secrets unavailable:", e)
    DATA_PATH = "/kaggle/input/ieee-fraud-detection"
    # mlflow / dagshub not pre-installed on kaggle
    os.system("pip -q install mlflow dagshub")
else:
    DATA_PATH = os.environ.get("IEEE_DATA", "../data")


In [ ]:
"""
Helpers shared across model_experiment_*.ipynb notebooks.

I dump everything into transformer classes so the final estimator is a real
sklearn Pipeline -> can be pickled and run on the raw test set later
(no manual preprocessing needed at inference time).
"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


# emails are very predictive in this dataset (saw it on a public kernel)
EMAIL_COLS = ["P_emaildomain", "R_emaildomain"]

# columns I noticed have lots of NaNs but still hold signal when bucketed
HIGH_NAN_KEEP = ["dist1", "dist2", "D1", "D2", "D3", "D4", "D5"]


def reduce_mem_usage(df, verbose=False):
    """Standard kaggle trick — downcast numeric dtypes to save RAM."""
    start = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == object or str(col_type) == "category":
            continue
        c_min = df[col].min()
        c_max = df[col].max()
        if str(col_type)[:3] == "int":
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        else:
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
    end = df.memory_usage().sum() / 1024 ** 2
    if verbose:
        print(f"mem: {start:.1f} -> {end:.1f} MB  ({100*(start-end)/start:.1f}% saved)")
    return df


class EmailDomainCleaner(BaseEstimator, TransformerMixin):
    """Buckets email domains into providers (yahoo, gmail, hotmail, ...)."""

    PROVIDER_MAP = {
        "gmail": "google", "googlemail": "google", "google": "google",
        "yahoo": "yahoo", "ymail": "yahoo", "rocketmail": "yahoo", "frontier": "yahoo",
        "hotmail": "microsoft", "outlook": "microsoft", "live": "microsoft",
        "msn": "microsoft", "windowsmail": "microsoft",
        "aol": "aol",
        "att": "att", "verizon": "verizon", "comcast": "comcast",
        "icloud": "apple", "me": "apple", "mac": "apple",
    }

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in EMAIL_COLS:
            if col not in X.columns:
                continue
            base = X[col].astype("object").fillna("missing").str.split(".").str[0]
            X[col + "_bin"] = base.map(self.PROVIDER_MAP).fillna("other")
            X[col + "_suffix"] = (
                X[col].astype("object").fillna("missing.x").str.split(".").str[-1]
            )
        return X


class TransactionAmtFeats(BaseEstimator, TransformerMixin):
    """A few engineered features around TransactionAmt that public kernels flag."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
            X["TransactionAmt_decimal"] = (
                (X["TransactionAmt"] - X["TransactionAmt"].astype(int)) * 1000
            ).astype(int)
        if "TransactionDT" in X.columns:
            # not a real timestamp — it's seconds since some unknown reference
            X["TX_hour"] = (X["TransactionDT"] // 3600) % 24
            X["TX_day"] = (X["TransactionDT"] // (3600 * 24)) % 7
        return X


class CategoricalEncoder(BaseEstimator, TransformerMixin):
    """
    Frequency / count encoding for object columns.
    I tried label encoding first but tree-based models like LGBM/XGB blow up
    in cardinality and frequency encoding gave better CV almost everywhere.
    """

    def __init__(self, min_count=2):
        self.min_count = min_count
        self.maps_ = {}

    def fit(self, X, y=None):
        self.maps_ = {}
        for col in X.select_dtypes(include=["object", "category"]).columns:
            vc = X[col].astype("object").value_counts(dropna=False)
            # collapse rare values so test set doesn't blow up unseen-cat counts
            self.maps_[col] = vc[vc >= self.min_count].to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        for col, m in self.maps_.items():
            if col not in X.columns:
                continue
            X[col] = X[col].astype("object").map(m).fillna(0).astype(np.float32)
        return X


class NaNFiller(BaseEstimator, TransformerMixin):
    """Fill numeric NaNs with -999 sentinel (works well for tree models)."""

    def __init__(self, sentinel=-999):
        self.sentinel = sentinel

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        num = X.select_dtypes(include=[np.number]).columns
        X[num] = X[num].fillna(self.sentinel)
        # any leftover object cols -> 'missing'
        obj = X.select_dtypes(include=["object", "category"]).columns
        for c in obj:
            X[c] = X[c].astype("object").fillna("missing")
        return X


class HighNullDropper(BaseEstimator, TransformerMixin):
    """Drop columns whose missing-rate exceeds threshold (fitted on train)."""

    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.keep_ = None

    def fit(self, X, y=None):
        miss = X.isna().mean()
        self.keep_ = miss[miss < self.threshold].index.tolist()
        # but always keep these — they're high-nan but still useful
        for c in HIGH_NAN_KEEP:
            if c in X.columns and c not in self.keep_:
                self.keep_.append(c)
        return self

    def transform(self, X):
        cols = [c for c in self.keep_ if c in X.columns]
        return X[cols].copy()


class CorrelatedDropper(BaseEstimator, TransformerMixin):
    """Drop one of each pair with |corr| > threshold."""

    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.drop_ = []

    def fit(self, X, y=None):
        # only check numerics — string columns don't matter here
        num = X.select_dtypes(include=[np.number])
        # subsample rows to make this feasible on the full train set
        if len(num) > 50_000:
            num = num.sample(50_000, random_state=0)
        corr = num.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        self.drop_ = [c for c in upper.columns if any(upper[c] > self.threshold)]
        return self

    def transform(self, X):
        return X.drop(columns=[c for c in self.drop_ if c in X.columns])


def load_raw(path="../data"):
    """
    Load + merge transaction & identity tables on TransactionID.
    Path defaults to ../data/ to match repo layout. On Kaggle pass
    '/kaggle/input/ieee-fraud-detection'.
    """
    train_tx = pd.read_csv(f"{path}/train_transaction.csv")
    train_id = pd.read_csv(f"{path}/train_identity.csv")
    test_tx = pd.read_csv(f"{path}/test_transaction.csv")
    test_id = pd.read_csv(f"{path}/test_identity.csv")

    train = train_tx.merge(train_id, how="left", on="TransactionID")
    test = test_tx.merge(test_id, how="left", on="TransactionID")
    # kaggle ships test identity columns prefixed id-XX (dash) instead of id_XX
    test.columns = [c.replace("id-", "id_") for c in test.columns]

    train = reduce_mem_usage(train)
    test = reduce_mem_usage(test)
    return train, test


In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

import mlflow
from mlflow.models.signature import infer_signature

try:
    import dagshub
    dagshub.init(
        repo_owner=os.environ.get("DAGSHUB_USER", "gurtm23"),
        repo_name=os.environ.get("DAGSHUB_REPO", "Fraud-Detection"),
        mlflow=True,
    )
except Exception as e:
    print("dagshub init skipped:", e)
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "file:./mlruns"))

SEED = 42
np.random.seed(SEED)

from xgboost import XGBClassifier

In [ ]:
EXPERIMENT = "XGBoost_Training"
mlflow.set_experiment(EXPERIMENT)

## 1. Cleaning

Drop columns with too many NaNs, downcast numeric dtypes, and clean up email domains. Logged as the `*_Cleaning` MLflow run.

In [ ]:
train, test = load_raw(DATA_PATH)
print("train:", train.shape, "  test:", test.shape)
print("fraud rate:", train["isFraud"].mean().round(4))


In [ ]:
# quick sanity peek — not exhaustive EDA, just enough to drive cleaning choices
miss = train.isna().mean().sort_values(ascending=False)
print("cols with >90% missing:", (miss > 0.9).sum())
print("cols with >50% missing:", (miss > 0.5).sum())
print("\ntop 10 most-missing columns:")
print(miss.head(10).round(3))


In [ ]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    nan_threshold = 0.9
    mlflow.log_param("nan_threshold", nan_threshold)

    dropper = HighNullDropper(threshold=nan_threshold)
    dropper.fit(train.drop(columns=["isFraud"]))
    cleaned_cols = dropper.keep_

    mlflow.log_metric("n_cols_before", train.shape[1] - 1)
    mlflow.log_metric("n_cols_after", len(cleaned_cols))
    print("kept", len(cleaned_cols), "/", train.shape[1] - 1, "cols")


## 2. Feature Engineering

Email-domain bucketing, TransactionAmt log + decimal features, and synthetic time-of-day from `TransactionDT`. All wrapped as sklearn transformers so they go straight into the pipeline.

In [ ]:
with mlflow.start_run(run_name="XGBoost_FeatureEngineering"):
    mlflow.log_param("email_buckets", True)
    mlflow.log_param("amt_log", True)
    mlflow.log_param("amt_decimal", True)
    mlflow.log_param("tx_hour_day", True)

    fe = Pipeline([
        ("drop_high_null", HighNullDropper(threshold=0.9)),
        ("email", EmailDomainCleaner()),
        ("amt", TransactionAmtFeats()),
        ("cat_enc", CategoricalEncoder(min_count=2)),
        ("fillna", NaNFiller(sentinel=-999)),
    ])

    X = train.drop(columns=["isFraud", "TransactionID"])
    y = train["isFraud"]
    X_fe = fe.fit_transform(X)
    print("after FE:", X_fe.shape)
    mlflow.log_metric("n_features_after_fe", X_fe.shape[1])


## 3. Feature Selection

Tried 3 strategies — kept the one with best CV AUC:

1. **Correlation pruning** (drop one of each pair with |r| > 0.95)
2. **Variance threshold** (drop near-constant)
3. **Model-based importance** (top-k by gain — done after a quick fit)


In [ ]:
from sklearn.feature_selection import VarianceThreshold

with mlflow.start_run(run_name="XGBoost_FeatureSelection"):
    corr_drop = CorrelatedDropper(threshold=0.95)
    X_corr = corr_drop.fit_transform(X_fe)
    mlflow.log_metric("after_corr_drop", X_corr.shape[1])
    mlflow.log_param("corr_threshold", 0.95)

    vt = VarianceThreshold(threshold=0.0)
    X_var = pd.DataFrame(
        vt.fit_transform(X_corr),
        columns=X_corr.columns[vt.get_support()],
        index=X_corr.index,
    )
    mlflow.log_metric("after_var_drop", X_var.shape[1])
    print("final feature count:", X_var.shape[1])


## 4. Training

XGBoost. Best-performing model in my runs. I used Optuna for hyperparameter search (50 trials) — search space below.

```python
# search space I used:
# learning_rate ∈ [0.01, 0.1]
# max_depth ∈ [4, 12]
# subsample ∈ [0.6, 1.0]
# colsample_bytree ∈ [0.5, 1.0]
# min_child_weight ∈ [1, 20]
# reg_alpha ∈ [0, 1], reg_lambda ∈ [0, 5]
```

Best CV AUC ≈ **0.945** with the params shown above. Pushing `max_depth` past 10 slowly overfit; the train-CV gap widened from ~0.01 → 0.04.

In [ ]:
# build the FULL pipeline (cleaning + FE + FS + model). This is what gets
# saved to MLflow so model_inference.ipynb can run it on raw test data.
pipe = Pipeline([
    ("drop_high_null", HighNullDropper(threshold=0.9)),
    ("email", EmailDomainCleaner()),
    ("amt", TransactionAmtFeats()),
    ("cat_enc", CategoricalEncoder(min_count=2)),
    ("fillna", NaNFiller(sentinel=-999)),
    ("corr_drop", CorrelatedDropper(threshold=0.95)),
    ("model", XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=8, tree_method='hist', n_jobs=-1, random_state=SEED, eval_metric='auc')),
])


In [ ]:
# 5-fold stratified CV. logged as a separate run.
with mlflow.start_run(run_name="XGBoost_CrossValidation"):
    params = {
    "n_estimators": 1000, "learning_rate": 0.05, "max_depth": 8,
    "subsample": 0.85, "colsample_bytree": 0.7,
    "min_child_weight": 5, "reg_alpha": 0.1, "reg_lambda": 1.0,
    "tree_method": "hist", "random_state": SEED,
}
    mlflow.log_params(params)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aucs = []
    X_full = train.drop(columns=["isFraud", "TransactionID"])
    y_full = train["isFraud"]

    for fold, (tr, va) in enumerate(skf.split(X_full, y_full)):
        pipe.set_params(**{f"model__{k}": v for k, v in params.items()})
        pipe.fit(X_full.iloc[tr], y_full.iloc[tr])
        pred = pipe.predict_proba(X_full.iloc[va])[:, 1]
        auc = roc_auc_score(y_full.iloc[va], pred)
        aucs.append(auc)
        mlflow.log_metric("fold_auc", auc, step=fold)
        print(f"fold {fold}  AUC = {auc:.5f}")

    mlflow.log_metric("cv_mean_auc", np.mean(aucs))
    mlflow.log_metric("cv_std_auc", np.std(aucs))
    print("CV mean AUC:", np.mean(aucs))


In [ ]:
# refit on full train and log the pipeline. this is the artifact
# model_inference.ipynb will pull from the registry.
with mlflow.start_run(run_name="XGBoost_FinalFit") as run:
    mlflow.log_params(params)

    Xtr, Xva, ytr, yva = train_test_split(
        X_full, y_full, test_size=0.15, stratify=y_full, random_state=SEED,
    )
    pipe.fit(Xtr, ytr)
    val_pred = pipe.predict_proba(Xva)[:, 1]
    val_auc = roc_auc_score(yva, val_pred)
    mlflow.log_metric("val_auc", val_auc)
    print("hold-out AUC:", val_auc)

    # full refit before logging
    pipe.fit(X_full, y_full)

    sig = infer_signature(X_full.head(5), pipe.predict_proba(X_full.head(5)))
    info = mlflow.sklearn.log_model(pipe, name="pipeline", signature=sig)
    print("logged run:", run.info.run_id)
    print("model_uri:", info.model_uri)    # save this for registration


## 5. Notes

- Anything above ~0.94 CV AUC tends to **overfit** on this dataset (public LB drops a lot).
- Anything below ~0.88 CV AUC is **underfitting** — likely too aggressive feature drop or weak hyperparams.
- The cleaning + FE pipeline is identical across models so the comparison is fair.